# FGAT - Notebook 2: Text Features + Graph Construction

**Mục tiêu:**
1. Extract text features bằng `bert-base-chinese` → `item_text_embs.npy`
2. Kết hợp visual + text → `item_embs.npy` (final item embeddings)
3. Tính category co-occurrence weights
4. Build 3 sparse edge matrices cho graph
5. Tạo train/val/test split files

**Input (từ Notebook 1 output + dataset):**
- `item_visual_embs.npy`, `item_id_order.npy`
- `item_data.txt`, `outfit_data.txt`, `user_data.txt`
- `train_uo.txt`, `test_uo.txt` (POG split files)

**Output:** tất cả file `.npy` và `.npz` cần cho Notebook 3

In [1]:
# ============================================================
# CELL 1: Install dependencies
# ============================================================
import subprocess
subprocess.run(['pip', 'install', '-q', 'transformers', 'torch', 'scipy', 'tqdm'], check=True)
print('Done')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 107.0 MB/s eta 0:00:00
Done


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

In [2]:
# ============================================================
# CELL 2: Config
# ============================================================
import os

# --- Input paths ---
DATA_DIR      = '/kaggle/input/datasets/kiettruonglifeez/recsys-fgat/'     # POG dataset
NB1_OUT_DIR   = '/kaggle/input/datasets/kiettruonglifeez/visual-extract-npy/' # output của Notebook 1
# Nếu chạy liên tiếp trong cùng session thì dùng:
# NB1_OUT_DIR = '/kaggle/working'

ITEM_FILE     = os.path.join(DATA_DIR, 'item_data.txt')
OUTFIT_FILE   = os.path.join(DATA_DIR, 'outfit_data.txt')
USER_FILE     = os.path.join(DATA_DIR, 'user_data.txt')
TRAIN_UO_FILE = os.path.join(DATA_DIR, 'train_uo.txt')
TEST_UO_FILE  = os.path.join(DATA_DIR, 'test_uo.txt')

VISUAL_EMB_FILE  = os.path.join(NB1_OUT_DIR, 'item_visual_embs.npy')
ITEM_ORDER_FILE  = os.path.join(NB1_OUT_DIR, 'item_id_order.npy')

# --- Output ---
OUT_DIR = '/kaggle/working'
os.makedirs(OUT_DIR, exist_ok=True)

# --- Dims ---
VISUAL_DIM  = 2048
TEXT_DIM    = 768
EMBED_DIM   = 64    # final embedding dim (sau projection)
BERT_BATCH  = 64
MAX_TXT_LEN = 64

print('Config OK')

Config OK


In [3]:
# ============================================================
# CELL 3: Load data
# ============================================================
import pandas as pd
import numpy as np

item_data   = pd.read_csv(ITEM_FILE,   header=None, names=['item_id','category','image_url','title'])
outfit_data = pd.read_csv(OUTFIT_FILE, header=None, names=['outfit_id','items'])
user_data = pd.read_csv(USER_FILE, header=None, names=['user_id', 'outfit_id'])

item_data['item_id']     = item_data['item_id'].astype(str)
outfit_data['outfit_id'] = outfit_data['outfit_id'].astype(str)
user_data['user_id']   = user_data['user_id'].astype(str)

# Load item_id order từ Notebook 1
item_id_order = np.load(ITEM_ORDER_FILE, allow_pickle=True).tolist()
item_to_idx   = {iid: i for i, iid in enumerate(item_id_order)}
outfit_ids    = outfit_data['outfit_id'].tolist()
outfit_to_idx = {oid: i for i, oid in enumerate(outfit_ids)}
# user_ids    = user_data['user_id'].unique().tolist()        Ở Cell 5c sẽ tạo từ train_uo.txt
# user_to_idx = {uid: i for i, uid in enumerate(user_ids)}

print(f'Items: {len(item_id_order)} | Outfits: {len(outfit_ids)}')
# | Users: {len(user_ids)

Items: 19175 | Outfits: 9373


In [4]:
# ============================================================
# Kiểm tra số lượng user trong user_data.txt vs train_uo.txt
# ============================================================
import pandas as pd

# --- user_data.txt ---
user_data = pd.read_csv(USER_FILE, header=None, names=['user_id', 'outfit_id'])
users_in_user_data = set(user_data['user_id'].astype(str).unique())

# --- train_uo.txt ---
users_in_train_uo = set()
with open(TRAIN_UO_FILE, 'r') as f:
    for line in f:
        uid = line.strip().split(' ')[0]
        if uid:
            users_in_train_uo.add(uid)

# --- So sánh ---
print(f'user_data.txt : {len(users_in_user_data):,} users')
print(f'train_uo.txt  : {len(users_in_train_uo):,} users')
print(f'Giống nhau    : {users_in_user_data == users_in_train_uo}')

# Chi tiết nếu khác nhau
only_in_user_data = users_in_user_data - users_in_train_uo
only_in_train_uo  = users_in_train_uo  - users_in_user_data

if only_in_user_data:
    print(f'Chỉ có trong user_data (không có trong train_uo): {len(only_in_user_data):,} users')
if only_in_train_uo:
    print(f'Chỉ có trong train_uo  (không có trong user_data): {len(only_in_train_uo):,} users')

user_data.txt : 277,469 users
train_uo.txt  : 277,469 users
Giống nhau    : True


In [5]:
# ============================================================
# CELL 4: Extract text features bằng bert-base-chinese
# ============================================================
import torch
from transformers import BertTokenizer, BertModel
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

tokenizer  = BertTokenizer.from_pretrained('bert-base-chinese')
bert_model = BertModel.from_pretrained('bert-base-chinese').to(device).eval()
print('BERT loaded')

class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len):
        self.texts     = texts
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx]) if pd.notna(self.texts[idx]) else ''
        enc  = self.tokenizer(text, return_tensors='pt',
                               padding='max_length', truncation=True,
                               max_length=self.max_len)
        return {k: v.squeeze(0) for k, v in enc.items()}

titles      = item_data['title'].tolist()
txt_dataset = TextDataset(titles, tokenizer, MAX_TXT_LEN)
txt_loader  = DataLoader(txt_dataset, batch_size=BERT_BATCH,
                          shuffle=False, num_workers=2, pin_memory=True)

text_embs = np.zeros((len(titles), TEXT_DIM), dtype=np.float32)

with torch.no_grad():
    for i, batch in enumerate(tqdm(txt_loader, desc='Extracting text features')):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = bert_model(**batch)
        # Average last 2 hidden layers [CLS] token - theo paper
        cls_emb = outputs.last_hidden_state[:, 0, :].cpu().numpy().astype(np.float32)
        start = i * BERT_BATCH
        end   = start + len(cls_emb)
        text_embs[start:end] = cls_emb

# Save
text_emb_path = os.path.join(OUT_DIR, 'item_text_embs.npy')
np.save(text_emb_path, text_embs)
print(f'Saved text embs: {text_embs.shape}')

Device: cuda


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/624 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT loaded


Extracting text features:   0%|          | 0/300 [00:00<?, ?it/s]

Saved text embs: (19175, 768)


In [6]:
# ============================================================
# CELL 5: Fuse visual + text → item_embs [N, 64]
# (dùng 1 Linear layer để project concat([visual, text]) -> 64)
# ============================================================
import torch.nn as nn

visual_embs = np.load(VISUAL_EMB_FILE)   # [N, 2048]
text_embs   = np.load(text_emb_path)     # [N, 768]

visual_t = torch.tensor(visual_embs, dtype=torch.float32)
text_t   = torch.tensor(text_embs,   dtype=torch.float32)

concat_dim = VISUAL_DIM + TEXT_DIM  # 2816
proj_layer = nn.Linear(concat_dim, EMBED_DIM)

# Initialize với xavier
nn.init.xavier_uniform_(proj_layer.weight)
nn.init.zeros_(proj_layer.bias)

with torch.no_grad():
    concat  = torch.cat([visual_t, text_t], dim=-1)  # [N, 2816]
    item_embs_proj = proj_layer(concat).numpy().astype(np.float32)  # [N, 64]

# Save
item_emb_path = os.path.join(OUT_DIR, 'item_embs.npy')
np.save(item_emb_path, item_embs_proj)
print(f'Saved item_embs: {item_embs_proj.shape}')

# Save projection layer weights (cần load lại khi train)
torch.save(proj_layer.state_dict(), os.path.join(OUT_DIR, 'item_proj_layer.pt'))
print('Saved projection layer weights')

Saved item_embs: (19175, 64)
Saved projection layer weights


In [7]:
# ============================================================
# CELL 5b: Build outfit_embs = mean of item_embs trong outfit
# ============================================================
outfit_embs_arr = np.zeros((len(outfit_ids), EMBED_DIM), dtype=np.float32)

for _, row in outfit_data.iterrows():
    oid   = str(row['outfit_id'])
    items = [it.strip() for it in str(row['items']).split(';')]
    if oid not in outfit_to_idx:
        continue
    o_idx  = outfit_to_idx[oid]
    i_idxs = [item_to_idx[it] for it in items if it in item_to_idx]
    if i_idxs:
        outfit_embs_arr[o_idx] = item_embs_proj[i_idxs].mean(axis=0)

outfit_emb_path = os.path.join(OUT_DIR, 'outfit_embs.npy')
np.save(outfit_emb_path, outfit_embs_arr)
print(f'Saved outfit_embs: {outfit_embs_arr.shape}')

Saved outfit_embs: (9373, 64)


In [8]:
# ============================================================
# CELL 5c: Build user_embs = mean of outfit_embs mà user đã tương tác
#          Nguồn: train_uo.txt (space-separated: user_id outfit1 outfit2 ...)
# ============================================================

# Đọc user list từ train_uo.txt
train_uo_users = []
train_uo_map   = {}   # user_id -> [outfit_idx, ...]

with open(TRAIN_UO_FILE, 'r') as f:
    for line in f:
        parts = line.strip().split(' ')
        uid   = parts[0]
        o_idxs = [outfit_to_idx[o] for o in parts[1:] if o in outfit_to_idx]
        if uid not in train_uo_map:
            train_uo_users.append(uid)
        train_uo_map[uid] = o_idxs

user_to_idx   = {uid: i for i, uid in enumerate(train_uo_users)}
user_ids      = train_uo_users

user_embs_arr = np.zeros((len(user_ids), EMBED_DIM), dtype=np.float32)

for uid, o_idxs in train_uo_map.items():
    if uid not in user_to_idx:
        continue
    u_idx = user_to_idx[uid]
    if o_idxs:
        user_embs_arr[u_idx] = outfit_embs_arr[o_idxs].mean(axis=0)

user_emb_path = os.path.join(OUT_DIR, 'user_embs.npy')
np.save(user_emb_path, user_embs_arr)

# Lưu thứ tự user_id (cần cho notebook 3)
np.save(os.path.join(OUT_DIR, 'user_id_order.npy'), np.array(user_ids))
print(f'Saved user_embs  : {user_embs_arr.shape}')
print(f'Saved user_id_order: {len(user_ids)} users')

Saved user_embs  : (277469, 64)
Saved user_id_order: 277469 users


In [9]:
# ============================================================
# CELL 6: User & Outfit initial embeddings (random)
# ============================================================
np.save(os.path.join(OUT_DIR, 'item_id_order.npy'),   np.array(item_id_order))
np.save(os.path.join(OUT_DIR, 'outfit_id_order.npy'), np.array(outfit_ids))
np.save(os.path.join(OUT_DIR, 'user_id_order.npy'),   np.array(user_ids))

print(f'item_id_order  : {len(item_id_order)}')
print(f'outfit_id_order: {len(outfit_ids)}')
print(f'user_id_order  : {len(user_ids)}')

item_id_order  : 19175
outfit_id_order: 9373
user_id_order  : 277469


In [10]:
# ============================================================
# CELL 7: Category co-occurrence weights
# ============================================================
from collections import defaultdict

item_category_map = dict(zip(item_data['item_id'].astype(str), item_data['category']))

category_cooccurrence = defaultdict(int)
category_count        = defaultdict(int)

for _, row in outfit_data.iterrows():
    items = [it.strip() for it in str(row['items']).split(';')]
    cats  = list(set(item_category_map[it] for it in items if it in item_category_map))
    for c in cats:
        category_count[c] += 1
    for i in range(len(cats)):
        for j in range(i+1, len(cats)):
            category_cooccurrence[(cats[i], cats[j])] += 1
            category_cooccurrence[(cats[j], cats[i])] += 1

# Tính weight theo công thức trong paper (Eq.6)
raw_weights = {}
for (c1, c2), g_cc in category_cooccurrence.items():
    g_c       = category_count[c1]
    denom     = sum(category_cooccurrence.get((c1, ck), 0) / max(category_count.get(ck, 1), 1)
                    for ck in category_count)
    raw_weights[(c1, c2)] = (g_cc / g_c) / denom if denom != 0 else 0.0

# Min-Max normalize
vals = list(raw_weights.values())
mn, mx = min(vals), max(vals)
norm_cat_weights = {
    k: (v - mn) / (mx - mn) if mx != mn else 0.0
    for k, v in raw_weights.items()
}

print(f'Category pairs: {len(norm_cat_weights)}')
print('Top 5:', sorted(norm_cat_weights.items(), key=lambda x: -x[1])[:5])

Category pairs: 1090
Top 5: [((19, 10), 1.0), ((19, 35), 1.0), ((19, 23), 1.0), ((57, 2), 0.4750608960412405), ((57, 5), 0.4750608960412405)]


In [11]:
# ============================================================
# CELL 8: Build item-item sparse matrix
# ============================================================
import scipy.sparse as sp
from tqdm.notebook import tqdm

N_items = len(item_id_order)
rows_ii, cols_ii, data_ii = [], [], []

for _, row in tqdm(outfit_data.iterrows(), total=len(outfit_data), desc='Building item-item edges'):
    items = [it.strip() for it in str(row['items']).split(';')]
    items = [it for it in items if it in item_to_idx]
    for i in range(len(items)):
        for j in range(i+1, len(items)):
            ci = item_category_map.get(items[i])
            cj = item_category_map.get(items[j])
            w  = norm_cat_weights.get((ci, cj), norm_cat_weights.get((cj, ci), 0.0))
            if w > 0:
                ii, jj = item_to_idx[items[i]], item_to_idx[items[j]]
                rows_ii += [ii, jj]
                cols_ii += [jj, ii]
                data_ii += [w, w]

item_item_mat = sp.csr_matrix((data_ii, (rows_ii, cols_ii)),
                               shape=(N_items, N_items), dtype=np.float32)
sp.save_npz(os.path.join(OUT_DIR, 'item_item_adj.npz'), item_item_mat)
print(f'item_item_adj: {item_item_mat.shape}, nnz={item_item_mat.nnz}')

Building item-item edges:   0%|          | 0/9373 [00:00<?, ?it/s]

item_item_adj: (19175, 19175), nnz=103652


In [12]:
# ============================================================
# CELL 9: Build outfit-item sparse matrix
# ============================================================
N_outfits = len(outfit_ids)
rows_oi, cols_oi = [], []

for _, row in outfit_data.iterrows():
    oid   = str(row['outfit_id'])
    items = [it.strip() for it in str(row['items']).split(';')]
    if oid not in outfit_to_idx:
        continue
    o_idx = outfit_to_idx[oid]
    for it in items:
        if it in item_to_idx:
            rows_oi.append(o_idx)
            cols_oi.append(item_to_idx[it])

outfit_item_mat = sp.csr_matrix(
    (np.ones(len(rows_oi), dtype=np.float32), (rows_oi, cols_oi)),
    shape=(N_outfits, N_items)
)
sp.save_npz(os.path.join(OUT_DIR, 'outfit_item_adj.npz'), outfit_item_mat)
print(f'outfit_item_adj: {outfit_item_mat.shape}, nnz={outfit_item_mat.nnz}')

outfit_item_adj: (9373, 19175), nnz=36401


In [13]:
# ============================================================
# CELL 10: Build user-outfit sparse matrix
# ============================================================
N_users = len(user_ids)
rows_uo, cols_uo = [], []

with open(TRAIN_UO_FILE, 'r') as f:
    for line in f:
        parts = line.strip().split(' ')  # ← space, không phải ';'
        uid   = parts[0]
        outfit_list = parts[1:]
        if uid not in user_to_idx:
            continue
        u_idx = user_to_idx[uid]
        for oid in outfit_list:
            if oid in outfit_to_idx:
                rows_uo.append(u_idx)
                cols_uo.append(outfit_to_idx[oid])

user_outfit_mat = sp.csr_matrix(
    (np.ones(len(rows_uo), dtype=np.float32), (rows_uo, cols_uo)),
    shape=(N_users, N_outfits)
)
sp.save_npz(os.path.join(OUT_DIR, 'user_outfit_adj.npz'), user_outfit_mat)
print(f'user_outfit_adj: {user_outfit_mat.shape}, nnz={user_outfit_mat.nnz}')

user_outfit_adj: (277469, 9373), nnz=679028


In [14]:
# ============================================================
# CELL 11: Tổng kết output files
# ============================================================
outputs = [
    'item_embs.npy', 'item_text_embs.npy',
    'user_embs.npy', 'outfit_embs.npy',
    'item_id_order.npy', 'outfit_id_order.npy', 'user_id_order.npy',
    'item_item_adj.npz', 'outfit_item_adj.npz', 'user_outfit_adj.npz',
    'item_proj_layer.pt'
]

print('Output files:')
for fn in outputs:
    path = os.path.join(OUT_DIR, fn)
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024**2
        print(f'  [OK] {fn:40s} {size:.1f} MB')
    else:
        print(f'  [MISSING] {fn}')

Output files:
  [OK] item_embs.npy                            4.7 MB
  [OK] item_text_embs.npy                       56.2 MB
  [OK] user_embs.npy                            67.7 MB
  [OK] outfit_embs.npy                          2.3 MB
  [OK] item_id_order.npy                        0.4 MB
  [OK] outfit_id_order.npy                      0.1 MB
  [OK] user_id_order.npy                        7.4 MB
  [OK] item_item_adj.npz                        0.4 MB
  [OK] outfit_item_adj.npz                      0.1 MB
  [OK] user_outfit_adj.npz                      1.6 MB
  [OK] item_proj_layer.pt                       0.7 MB
